In [5]:
# # Linear regression 

# # x - price 
# # y - date

# import datetime
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from pandas_datareader import data
# from sklearn.metrics import mean_squared_error

# tickers = ['AAPL']

# start_date = '2019-01-01'

# today = datetime.date.today()
# yesterday = today - datetime.timedelta(days=1)
# end_date = yesterday

# # User pandas_reader.data.DataReader to load data
# panel_data = data.DataReader(tickers, 'yahoo', start_date, end_date)


# # df = panel_data['Close']
# df = panel_data
# df.head()

Attributes,Adj Close,Close,High,Low,Open,Volume
Symbols,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,,
2018-12-31,38.282612,39.435001,39.840000,39.119999,39.632500,140014000.0
2019-01-02,38.326294,39.480000,39.712502,38.557499,38.722500,148158800.0
2019-01-03,34.508713,35.547501,36.430000,35.500000,35.994999,365248800.0
2019-01-04,35.981865,37.064999,37.137501,35.950001,36.132500,234428400.0
2019-01-07,35.901775,36.982498,37.207500,36.474998,37.174999,219111200.0


In [44]:
from sklearn.linear_model import LinearRegression
from fastai.structured import  add_datepart

def linear_regression(ticker):
    
    #setting index as date values
    df['Date'] = pd.to_datetime(df.Date,format='%Y-%m-%d')
    df.index = df['Date']

    #sorting
    data = df.sort_index(ascending=True, axis=0)

    #creating a separate dataset
    new_data = pd.DataFrame(index=range(0,len(df)),columns=['Date', 'Close'])

    for i in range(0,len(data)):
        new_data['Date'][i] = data['Date'][i]
        new_data['Close'][i] = data['Close'][i]

    #create features
    add_datepart(new_data, 'Date')
    new_data.drop('Elapsed', axis=1, inplace=True)  #elapsed will be the time stamp

    new_data['mon_fri'] = 0
    for i in range(0,len(new_data)):
        if (new_data['Dayofweek'][i] == 0 or new_data['Dayofweek'][i] == 4):
            new_data['mon_fri'][i] = 1
        else:
            new_data['mon_fri'][i] = 0

    train = new_data[:987]
    valid = new_data[987:]

    x_train = train.drop('Close', axis=1)
    y_train = train['Close']
    x_valid = valid.drop('Close', axis=1)
    y_valid = valid['Close']

    model = LinearRegression()
    model.fit(x_train,y_train)
    
    #make predictions and find the rmse
    preds = model.predict(x_valid)
    rms=np.sqrt(np.mean(np.power((np.array(y_valid)-np.array(preds)),2)))
    rms

    #plot
    valid['Predictions'] = 0
    valid['Predictions'] = preds

    valid.index = new_data[987:].index
    train.index = new_data[:987].index

    plt.plot(train['Close'])
    plt.plot(valid[['Close', 'Predictions']])

    return price 

ModuleNotFoundError: No module named 'fastai.structured'

In [45]:
!pip install fastai.structured

ERROR: Could not find a version that satisfies the requirement fastai.structured
ERROR: No matching distribution found for fastai.structured


In [267]:
test = linear_regression("VXRT")
test

UnboundLocalError: local variable 'data' referenced before assignment

In [268]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas_datareader import data
from sklearn.metrics import mean_squared_error
from fastai.tabular.all import *
import datetime

def linear_regression(stockname):
    global data
    tickers = [stockname]

    start_date = '2019-01-01'
    end_date = datetime.date.today() 

    # User pandas_reader.data.DataReader to load data
    panel_data = data.DataReader(tickers,'yahoo', start_date, end_date)

    # df = panel_data['Close']
    df = panel_data

    panel_data["Date"] = panel_data.index
    panel_data["Date"] = pd.to_datetime(panel_data['Date'])

    df['Date'] = pd.to_datetime(df.Date,format='%Y-%m-%d')
    df.index = df['Date']

    #sorting
    data = df.sort_index(ascending=True, axis=0)

    #creating a separate dataset
    new_data = pd.DataFrame(index=range(0,len(df)),columns=['Date', 'Close'])



    for i in range(0,len(data)):
        new_data['Date'][i] = data['Date'][i]
        new_data['Close'][i] = data['Close'].iloc[i][stockname]

    #create features
    add_datepart(new_data, 'Date')
    new_data.drop('Elapsed', axis=1, inplace=True)  #elapsed will be the time stamp


    new_data['mon_fri'] = 0
    new_data
    for i in range(0,len(new_data)):
        if (new_data['Dayofweek'][i] == 0 or new_data['Dayofweek'][i] == 4):
            new_data.at[i,'mon_fri'] = 1
        else:
            new_data.at[i,'mon_fri'] = 0


    end = int(len(new_data) * 0.8)

    train = new_data[:end]
    valid = new_data[end:]

    valid

    x_train = train.drop('Close', axis=1)
    y_train = train['Close']
    x_valid = valid.drop('Close', axis=1)
    y_valid = valid['Close']

    model = LinearRegression()
    model.fit(x_train,y_train)


    #make predictions and find the rmse
    preds = model.predict(x_valid)
    rms=np.sqrt(np.mean(np.power((np.array(y_valid)-np.array(preds)),2)))


    yst_price = panel_data.iloc[-2]["Close"][stockname]

    tdy = pd.DataFrame([datetime.date.today()],columns=["Date"])

    add_datepart(tdy, 'Date')
    tdy.drop('Elapsed', axis=1, inplace=True)  #elapsed will be the time stamp
    tdy['mon_fri'] = 1
    if (tdy['Dayofweek'][0] == 0 or tdy['Dayofweek'][0] == 4):
        tdy['mon_fri'] = 1
    else:
        tdy['mon_fri'] = 0


    tdy_preds = model.predict(tdy)
    tdy_preds
    json_result = {'today_price': tdy_preds, 'yesterday_price':yst_price, 'rmse': rms}
    return json_result

In [269]:
test = linear_regression("VXRT")
test

{'today_price': array([9.45888358]),
 'yesterday_price': 6.889999866485596,
 'rmse': 1.2328488081248794}

In [66]:
import time
import datetime
from flask import Flask
from flask import jsonify
import pandas as pd
import statistics
from collections import Counter

# libraries for Stocktwits
import requests
import json
import os

# libraries for News
from GoogleNews import GoogleNews
from newspaper import Article
from newspaper import Config


# libraries for Reddit
from psaw import PushshiftAPI

# libraries for clean
import re
import nltk
from nltk import word_tokenize
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import numpy as np

# libraries for sentiment analysis
import flair
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from happytransformer import HappyTextClassification

# libraries for prediction
from pandas_datareader import data
from sklearn.metrics import mean_squared_error
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA
from fbprophet import Prophet
from sklearn.preprocessing import MinMaxScaler
# from keras.models import Sequential
# from keras.layers import Dense, Dropout, LSTM

def autoArimaML(symbol, df):
    close = df['Close']

    # splitting the data
    train_ratio = 0.8
    train_size = int(df.shape[0] * train_ratio)
    test_size = df.shape[0] - train_size

    train = close.head(train_size)
    test = close.tail(test_size)

    model = auto_arima(train, start_p=1, start_q=1,max_p=6, max_q=3, m=12, seasonal=True, error_action='ignore',suppress_warnings=True)
    model.fit(train)

    forecast = model.predict(n_periods=test_size)
    
    todayPredict = forecast[-1]
    ytdClose = test[symbol][-1]
    fivedaypredicted = test[symbol][-5:]
    MSE_error = mean_squared_error(test, forecast)


    graphData = [] 
    for index, row in train.iterrows():
        dateobj = {"date":str(index.date()), "train":(row[0]) }
        graphData.append(dateobj)

    i = 0
    for index, row in test.iterrows():
        dateobj = {"date":str(index.date()), "test":(row[0]), "predicted":forecast[i] }
        graphData.append(dateobj)
        i += 1
     
    return todayPredict, ytdClose, MSE_error, graphData

start_date = '2019-01-01'

today = datetime.date.today()
yesterday = today - datetime.timedelta(days=1)
end_date = yesterday

panel_data = data.DataReader(["AAPL"], 'yahoo', start_date, end_date)
test = autoArimaML("AAPL", panel_data)
print(test)

 


ModuleNotFoundError: No module named 'fbprophet'

In [76]:
from fbprophet import Prophet

ModuleNotFoundError: No module named 'fbprophet'

In [77]:
# from prophet import Prophet
# !pip install pystan==2.19.1.1 prophet
conda create -n time_series python=3.8


SyntaxError: invalid syntax (<ipython-input-77-c361299fab77>, line 3)

In [64]:
def arima(symbol, df):
    # preprocessing data
    train_data, test_data = df[0:int(len(df)*0.8)], df[int(len(df)*0.8):]
    training_data = train_data['Close'].values
    graph_data = test_data['Close'] 
    test_data = test_data['Close'].values

    history = [x for x in training_data]
    model_predictions = []
    N_test_observations = len(test_data)
    yhat = 0
    # train model
    for time_point in range(N_test_observations):
        try:
            model = ARIMA(history, order=(4,1,0))
            model_fit = model.fit()
            output = model_fit.predict()
            yhat = output[-1]
            true_test_value = test_data[time_point]
            history.append(true_test_value)
            model_predictions.append(yhat)
        except Exception as e:
            print(e)
            return 0, test_data[-1], 0

    model_pred = pd.DataFrame(model_predictions,columns=['Prediction'])
    test = pd.DataFrame(test_data, columns = [symbol])

    # calculating rmse
    MSE_error = mean_squared_error(test, model_pred)


    graphData = [] 
    for index, row in train_data['Close'].iterrows():
        
        dateobj = {"date":str(index.date()), "train":row[0] }

        graphData.append(dateobj)

    i = 0
    for index, row in graph_data.iterrows():
        dateobj = {"date":str(index.date()), "test":row[0], "predicted":model_predictions[i] }
        graphData.append(dateobj)
        i += 1
        

    return model_predictions[-1], test_data[-1], MSE_error,graphData


tset = arima("AAPL",panel_data)
tset

[{'date': '2018-12-31', 'train': 39.435001373291016},
 {'date': '2019-01-02', 'train': 39.47999954223633},
 {'date': '2019-01-03', 'train': 35.54750061035156},
 {'date': '2019-01-04', 'train': 37.064998626708984},
 {'date': '2019-01-07', 'train': 36.98249816894531},
 {'date': '2019-01-08', 'train': 37.6875},
 {'date': '2019-01-09', 'train': 38.32749938964844},
 {'date': '2019-01-10', 'train': 38.45000076293945},
 {'date': '2019-01-11', 'train': 38.0724983215332},
 {'date': '2019-01-14', 'train': 37.5},
 {'date': '2019-01-15', 'train': 38.26750183105469},
 {'date': '2019-01-16', 'train': 38.73500061035156},
 {'date': '2019-01-17', 'train': 38.96500015258789},
 {'date': '2019-01-18', 'train': 39.20500183105469},
 {'date': '2019-01-22', 'train': 38.32500076293945},
 {'date': '2019-01-23', 'train': 38.47999954223633},
 {'date': '2019-01-24', 'train': 38.17499923706055},
 {'date': '2019-01-25', 'train': 39.439998626708984},
 {'date': '2019-01-28', 'train': 39.07500076293945},
 {'date': '201

In [68]:
def prophet(symbol, df):
    close = df['Close']

    #preparing data
    close.reset_index(inplace = True)
    close = close[['Date', symbol]]
    close.rename(columns={symbol: 'y', 'Date':'ds'}, inplace = True)

    #splitting the data
    train_ratio = 0.8
    train_size = int(df.shape[0] * train_ratio)
    test_size = df.shape[0] - train_size

    train = close.head(train_size)
    test = close.tail(test_size)

    #fit the model
    model = Prophet(daily_seasonality=True)
    model.fit(train)

    #predictions
    close_prices = model.make_future_dataframe(periods=len(test)+1)
    forecast = model.predict(close_prices)
    today_prediction = forecast['yhat'].to_list()[-1]

    # combine test and forecast dataframes
    forecast = pd.DataFrame(forecast,index = test.index,columns=['yhat'])
    result = pd.concat([forecast, test], axis = 1)

    #rmse
    forecast_valid = forecast['yhat'][-test_size:]
    MSE_error = mean_squared_error(test['y'], forecast_valid)

    # renaming columns in result
    result.rename(columns = {'yhat':'Prediction', 'y': symbol}, inplace = True)
    result.drop(columns = ['ds'], inplace = True)
    
    graphData = [] 
    for index, row in train_data['Close'].iterrows():
        
        dateobj = {"date":str(index.date()), "train":row[0] }

        graphData.append(dateobj)

    i = 0
    for index, row in graph_data.iterrows():
        dateobj = {"date":str(index.date()), "test":row[0], "predicted":model_predictions[i] }
        graphData.append(dateobj)
        i += 1
    return graphData

    return today_prediction, test['y'].to_list()[-1], MSE_error


tset = prophet("AAPL",panel_data)
tset

NameError: name 'Prophet' is not defined

In [84]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM

In [85]:
# !pip install keras
# !pip install tensorflow

In [99]:
def lstm(symbol, df):
    close = df['Close']

    #preparing data
    close.reset_index(inplace = True)
    close = close[['Date', symbol]]

    #splitting the data
    train_ratio = 0.8
    train_size = int(df.shape[0] * train_ratio)
    test_size = df.shape[0] - train_size

    train = close.head(train_size)
    test = close.tail(-test_size)

    close.drop('Date', axis=1, inplace = True)

    #convert into x and y
    scaler = MinMaxScaler(feature_range=(0,1))
    scaled_data = scaler.fit_transform(close)

    x_train, y_train = [], []
    for i in range(60,len(train)):
        x_train.append(scaled_data[i-60:i,0])
        y_train.append(scaled_data[i,0])
    x_train, y_train = np.array(x_train), np.array(y_train)
    x_train = np.reshape(x_train, (x_train.shape[0],x_train.shape[1],1))

    # create and fit the LSTM network
    model = Sequential()
    model.add(LSTM(units=50, return_sequences=True, input_shape=(x_train.shape[1],1)))
    model.add(LSTM(units=50))
    model.add(Dense(1))

    model.compile(loss='mean_squared_error', optimizer='adam')
    model.fit(x_train, y_train, epochs=1, batch_size=1, verbose=2)

    #prediction
    inputs = close[len(close) - len(test) - 60:].values
    inputs = inputs.reshape(-1,1)
    inputs  = scaler.transform(inputs)

    X_test = []
    for i in range(60,inputs.shape[0]+1):
        X_test.append(inputs[i-60:i,0])
    X_test = np.array(X_test)

    X_test = np.reshape(X_test, (X_test.shape[0],X_test.shape[1],1))
    closing_price = model.predict(X_test)
    closing_price = scaler.inverse_transform(closing_price)
    today_prediction = closing_price[-1]
    closing_price = closing_price[:-1]

    graphData = [] 
    
    for index, row in train.iterrows():
        dateobj = {"date":str(row[0].date()), "train":row[1] }

        graphData.append(dateobj)
    i = 0
    for index, row in test.iterrows():
        dateobj = {"date":str(row[0].date()), "test":row[1], "predicted":closing_price[i] }
        graphData.append(dateobj)
        i += 1
    return graphData

    #rmse
    MSE_error = mean_squared_error(test[symbol], closing_price)

    test['Predictions'] = closing_price
    test.drop(columns = ['Date'], inplace = True)

    return today_prediction[0], closing_price[-1][0], MSE_error


test = lstm("AAPL",panel_data)
test

517/517 - 7s - loss: 0.0047 - 7s/epoch - 14ms/step


[{'date': '2018-12-31', 'train': 39.435001373291016},
 {'date': '2019-01-02', 'train': 39.47999954223633},
 {'date': '2019-01-03', 'train': 35.54750061035156},
 {'date': '2019-01-04', 'train': 37.064998626708984},
 {'date': '2019-01-07', 'train': 36.98249816894531},
 {'date': '2019-01-08', 'train': 37.6875},
 {'date': '2019-01-09', 'train': 38.32749938964844},
 {'date': '2019-01-10', 'train': 38.45000076293945},
 {'date': '2019-01-11', 'train': 38.0724983215332},
 {'date': '2019-01-14', 'train': 37.5},
 {'date': '2019-01-15', 'train': 38.26750183105469},
 {'date': '2019-01-16', 'train': 38.73500061035156},
 {'date': '2019-01-17', 'train': 38.96500015258789},
 {'date': '2019-01-18', 'train': 39.20500183105469},
 {'date': '2019-01-22', 'train': 38.32500076293945},
 {'date': '2019-01-23', 'train': 38.47999954223633},
 {'date': '2019-01-24', 'train': 38.17499923706055},
 {'date': '2019-01-25', 'train': 39.439998626708984},
 {'date': '2019-01-28', 'train': 39.07500076293945},
 {'date': '201

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas_datareader import data
from sklearn.metrics import mean_squared_error
from fastai.tabular.all import *
import datetime

def linear_regression(stockname):
    global data
    tickers = [stockname]

    start_date = '2019-01-01'
    end_date = datetime.date.today() 

    # User pandas_reader.data.DataReader to load data
    panel_data = data.DataReader(tickers,'yahoo', start_date, end_date)

    # df = panel_data['Close']
    df = panel_data

    panel_data["Date"] = panel_data.index
    panel_data["Date"] = pd.to_datetime(panel_data['Date'])

    df['Date'] = pd.to_datetime(df.Date,format='%Y-%m-%d')
    df.index = df['Date']

    #sorting
    data = df.sort_index(ascending=True, axis=0)

    #creating a separate dataset
    new_data = pd.DataFrame(index=range(0,len(df)),columns=['Date', 'Close'])



    for i in range(0,len(data)):
        new_data['Date'][i] = data['Date'][i]
        new_data['Close'][i] = data['Close'].iloc[i][stockname]

    #create features
    add_datepart(new_data, 'Date')
    new_data.drop('Elapsed', axis=1, inplace=True)  #elapsed will be the time stamp


    new_data['mon_fri'] = 0
    new_data
    for i in range(0,len(new_data)):
        if (new_data['Dayofweek'][i] == 0 or new_data['Dayofweek'][i] == 4):
            new_data.at[i,'mon_fri'] = 1
        else:
            new_data.at[i,'mon_fri'] = 0


    end = int(len(new_data) * 0.8)

    train = new_data[:end]
    valid = new_data[end:]

    valid

    x_train = train.drop('Close', axis=1)
    y_train = train['Close']
    x_valid = valid.drop('Close', axis=1)
    y_valid = valid['Close']

    model = LinearRegression()
    model.fit(x_train,y_train)


    #make predictions and find the rmse
    preds = model.predict(x_valid)
    rms=np.sqrt(np.mean(np.power((np.array(y_valid)-np.array(preds)),2)))


    yst_price = panel_data.iloc[-2]["Close"][stockname]

    tdy = pd.DataFrame([datetime.date.today()],columns=["Date"])

    add_datepart(tdy, 'Date')
    tdy.drop('Elapsed', axis=1, inplace=True)  #elapsed will be the time stamp
    tdy['mon_fri'] = 1
    if (tdy['Dayofweek'][0] == 0 or tdy['Dayofweek'][0] == 4):
        tdy['mon_fri'] = 1
    else:
        tdy['mon_fri'] = 0


    tdy_preds = model.predict(tdy)
    tdy_preds
    json_result = {'today_price': tdy_preds, 'yesterday_price':yst_price, 'rmse': rms}
    return json_result